# Convert and Optimize BoT-SORT multi-object tracking with OpenVINO™

[BoT-SORT](https://github.com/NirAharon/BoT-SORT) ([paper](https://arxiv.org/abs/2206.14651)) tracks people with **two
neural networks** plus a host-side tracker:
* a **YOLOX-X detector** (800×1440, 790 GFLOP per frame) finds the people,
* a **FastReID SBS-S50 ReID network** (ResNeSt-50, 14 GFLOP per crop) embeds each confident detection (2048-d),
* the **BoT-SORT tracker** compensates camera motion, predicts with a Kalman filter and associates detections to tracks
  using IoU fused with appearance similarity.

This tutorial converts and optimizes **both networks** with OpenVINO, end to end from a fresh clone:
1. Clone the BoT-SORT repository and download the two official checkpoints; build the **PyTorch FP32 models**.
2. Convert both to **OpenVINO IR FP32**, then the FP32 IRs to **FP16**.
3. Verify parity on a sample frame; optionally check accuracy (MOTA / IDF1) on MOT17 val-half if you have the dataset.
4. Run both precisions on a free-license sample video, scored against **PyTorch FP32 reference tracks** computed here.
5. Quantize FP32 IR → **INT8** with NNCF post-training quantization. If the accuracy drops by more than 2 points, use
   `nncf.quantize_with_accuracy_control`.
6. Run INT8 on the video and measure **average latency and FPS with `benchmark_app`**.
7. Compare every OpenVINO precision with the PyTorch FP32 baseline.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Get PyTorch models and tracking config](#Get-PyTorch-models-and-tracking-config)
- [Convert models to OpenVINO IR](#Convert-models-to-OpenVINO-IR)
    - [FP32 IR](#FP32-IR)
    - [FP32 IR to FP16 IR](#FP32-IR-to-FP16-IR)
- [Select inference device](#Select-inference-device)
- [Verify model inference](#Verify-model-inference)
- [Accuracy of OpenVINO FP32 and FP16 (optional, MOT17)](#Accuracy-of-OpenVINO-FP32-and-FP16-(optional,-MOT17))
- [Run FP32 and FP16 on a video](#Run-FP32-and-FP16-on-a-video)
- [Optimize models using NNCF Post-training Quantization API](#Optimize-models-using-NNCF-Post-training-Quantization-API)
    - [Validate Quantized model inference](#Validate-Quantized-model-inference)
    - [Run INT8 on the video](#Run-INT8-on-the-video)
- [Compare performance of the models with benchmark_app](#Compare-performance-of-the-models-with-benchmark_app)
- [Accuracy and performance: OpenVINO vs PyTorch](#Accuracy-and-performance:-OpenVINO-vs-PyTorch)

### Installation Instructions

This is a self-contained example that relies solely on its own code (`botsort_nb_utils.py` next to the notebook).
We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

* PyTorch for your device first (Intel GPU: `pip install torch torchvision --index-url https://download.pytorch.org/whl/xpu`),
  then `pip install openvino nncf opencv-python motmetrics lap yacs ipywidgets matplotlib gdown imageio-ffmpeg`.
* The BoT-SORT repository and the two official checkpoints are downloaded automatically on first run. To reuse local
  copies, set the environment variables `BOTSORT_REPO`, `BOTSORT_DET_WEIGHTS`, `BOTSORT_REID_WEIGHTS`.
* The sample video is downloaded from Wikimedia Commons
  (CC BY-SA 4.0, "Pro dog walker - Tokyo - 2024 Nov 1" by User:Nesnad) and transcoded to H.264.
* The MOT17 dataset is **optional**: it unlocks the MOTA / IDF1 accuracy sections. Set `MOT17_TRAIN` to its `train`
  folder. Without it, the video demo, quantization and benchmarks still run end to end.


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# %pip install -q -r requirements.txt
import os, sys, json, time, shutil
from pathlib import Path
import numpy as np, cv2, torch
import openvino as ov
import nncf
from IPython.display import Image, Markdown, display

import botsort_nb_utils as U

core = ov.Core()
print("OpenVINO", ov.__version__, "| NNCF", nncf.__version__)
print("devices:", {d: core.get_property(d, "FULL_DEVICE_NAME") for d in core.available_devices})
OUT = Path("results")
(OUT / "images").mkdir(parents=True, exist_ok=True)

## Get PyTorch models and tracking config
[back to top ⬆️](#Table-of-contents:)

The cell below clones the unmodified BoT-SORT repository (the tracker and the vendored FastReID code), downloads the
two official checkpoints if they are not present, and builds the FP32 PyTorch networks from them. The tracking
configuration ships with `botsort_nb_utils.py`.

**Is a tracking config needed for OpenVINO?** Yes. Detection / NMS thresholds, the ReID crop pre-processing, the tracker
thresholds and ReID gates, the track buffer with its per-sequence overrides and the camera-motion settings are in
neither weight file nor IR graph. OpenVINO only replaces the networks, so the same config is used. It is embedded into
both IRs' `rt_info` and saved next to them.


In [ ]:
U.ensure_repo()
U.setup_environment()
U.download_weights()
cfg = U.DEFAULT_CONFIG
det_pt = U.build_detector(U.DET_WEIGHTS)
reid_pt = U.build_reid(U.REID_WEIGHTS)
DET_SHAPE = [1, 3, *cfg["detector"]["input_size_hw"]]
REID_SHAPE = [cfg["reid"]["batch"], 3, *cfg["reid"]["input_hw"]]
acc = {}

MOT17_OK = U.mot17_available()
print("MOT17 dataset:", "found - accuracy sections enabled" if MOT17_OK else "not found - accuracy sections will be skipped (set MOT17_TRAIN to enable)")

# evaluation / benchmark settings; full_evaluation=True evaluates every MOT17 val-half sequence in full
SETTINGS = {"full_evaluation": False, "accuracy_sequences": None, "bench_time_s": 30, "cooldown_s": 10}

## Convert models to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). `ov.convert_model`
traces each FP32 module and `ov.save_model` serializes it as an **FP32 IR**. The **FP16 IRs** are made from the
**saved FP32 IRs** with `compress_to_fp16=True`: weights become FP16, inputs/outputs stay f32.

The ReID IR has a **static batch of 8** (the official FastReID batch size). Partial batches are zero-padded, as in the
original implementation.

### FP32 IR
[back to top ⬆️](#Table-of-contents:)

In [ ]:
IR_DIR = Path("ov_models")
DET_NAME, REID_NAME = "botsort_yolox_x_detector.xml", "botsort_sbs_s50_reid.xml"
ir = lambda variant, name: IR_DIR / variant / name  # noqa: E731
tracking_config = json.dumps(cfg)


def save_ir(model: ov.Model, xml: Path, compress_to_fp16: bool) -> None:
    model.set_rt_info(tracking_config, ["tracking_config"])  # the tracking config travels with the IR
    xml.parent.mkdir(parents=True, exist_ok=True)
    ov.save_model(model, str(xml), compress_to_fp16=compress_to_fp16)
    json.dump(cfg, open(xml.parent / "botsort_config.json", "w"), indent=2)


for name, module, shape, in_name in ((DET_NAME, det_pt, DET_SHAPE, "images"), (REID_NAME, reid_pt, REID_SHAPE, "crops")):
    if not ir("FP32", name).exists():
        t = time.perf_counter()
        m = ov.convert_model(module, example_input=torch.zeros(shape), input=[ov.PartialShape(shape)])
        m.inputs[0].get_tensor().set_names({in_name})
        save_ir(m, ir("FP32", name), compress_to_fp16=False)
        print(f"{name}: converted in {time.perf_counter() - t:.1f}s")
    print(f"FP32 IR {ir('FP32', name)} ({ir('FP32', name).with_suffix('.bin').stat().st_size / 1e6:.0f} MB)")

### FP32 IR to FP16 IR
[back to top ⬆️](#Table-of-contents:)

In [ ]:
for name in (DET_NAME, REID_NAME):
    if not ir("FP16", name).exists():
        save_ir(core.read_model(ir("FP32", name)), ir("FP16", name), compress_to_fp16=True)
    m = core.read_model(ir("FP16", name))
    print(f"FP16 IR {ir('FP16', name)} ({ir('FP16', name).with_suffix('.bin').stat().st_size / 1e6:.0f} MB) | "
          f"input {m.input(0).get_element_type()} {m.input(0).get_partial_shape()} | rt_info tracking_config: {m.has_rt_info(['tracking_config'])}")

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select the device from the dropdown list for running inference with OpenVINO.

> **FP32 on GPU:** the GPU plugin executes an FP32 IR in **f16** by default. True FP32, comparable to PyTorch FP32,
> needs `INFERENCE_PRECISION_HINT=f32` (`benchmark_app -infer_precision f32`). All "OV FP32" numbers below use it.

In [ ]:
import ipywidgets as widgets

device = widgets.Dropdown(options=core.available_devices + ["AUTO"], value="GPU" if "GPU" in core.available_devices else "CPU", description="Device:")
device

## Verify model inference
[back to top ⬆️](#Table-of-contents:)

One frame of the sample video through both detector IRs and one batch of its person crops through both ReID IRs,
compared with PyTorch FP32 on CPU: cosine of the raw detector output and of the score map (obj × cls), number of
detections, and cosine of the ReID embeddings.


In [ ]:
FP32_PROPS = {"INFERENCE_PRECISION_HINT": "f32"} if device.value.startswith("GPU") else {}
det_ov = {"FP32": U.ov_runner(core, ir("FP32", DET_NAME), device.value, **FP32_PROPS), "FP16": U.ov_runner(core, ir("FP16", DET_NAME), device.value)}
reid_ov = {"FP32": U.ov_runner(core, ir("FP32", REID_NAME), device.value, **FP32_PROPS), "FP16": U.ov_runner(core, ir("FP16", REID_NAME), device.value)}

U.ensure_video()
img0 = U.video_frames(U.VIDEO_1, 1)[0]
x0, r0 = U.preprocess(img0, cfg)
with torch.inference_mode():
    det_ref = det_pt(torch.from_numpy(x0)).numpy()
d_ref = U.postprocess(det_ref, r0, cfg)
crops0, n0 = next(U.batched(U.reid_crops(img0, d_ref[d_ref[:, 4] > 0.6], tuple(cfg["reid"]["input_hw"])), cfg["reid"]["batch"]))
with torch.inference_mode():
    emb_ref = U.normalize_feats(reid_pt(torch.from_numpy(crops0)).numpy()[:n0])
print(f"PyTorch FP32 (CPU): {len(d_ref)} detections")
for p in ("FP32", "FP16"):
    y, ms = U.timeit(det_ov[p], x0, n=10)
    e, ms_r = U.timeit(reid_ov[p], crops0, n=10)
    emb = U.normalize_feats(e[:n0])
    print(f"OV {p} (exec {det_ov[p].exec_precision}): detector {ms:6.1f} ms, cosine raw {U.cosine(y, det_ref):.7f} / scores {U.score_cosine(y, det_ref):.7f}, "
          f"{len(U.postprocess(y, r0, cfg))} detections | ReID {ms_r:5.1f} ms/batch, embedding cosine {np.mean(np.sum(emb * emb_ref, 1)):.6f}")

## Accuracy of OpenVINO FP32 and FP16 (optional, MOT17)
[back to top ⬆️](#Table-of-contents:)

MOTA / IDF1 of the full pipeline on MOT17 val-half (the ByteTrack / BoT-SORT ablation protocol: second half of each
train FRCNN sequence, ground truth filtered to pedestrians with visibility ≥ 0.25 and ignore regions removed).
This section runs only when the MOT17 dataset is available; the rest of the notebook does not need it.
By default one sequence is evaluated on its first 150 val-half frames for a quick check — set
`SETTINGS["full_evaluation"] = True` for the full protocol.


In [ ]:
if MOT17_OK:
    ACC_SEQS = SETTINGS["accuracy_sequences"] or ["MOT17-09-FRCNN"]
    ACC_FRAMES = None if SETTINGS["full_evaluation"] else 150
    det_pt_run = U.torch_runner(det_pt, "cpu", torch.float32)
    reid_pt_run = U.torch_runner(reid_pt, "cpu", torch.float32)
    acc["PT_FP32"] = U.mot_eval(det_pt_run, reid_pt_run, cfg, "PT_FP32", seqs=ACC_SEQS, max_frames=ACC_FRAMES)
    for p in ("FP32", "FP16"):
        acc[f"OV_{p}"] = U.mot_eval(det_ov[p], reid_ov[p], cfg, f"OV_{p}", seqs=ACC_SEQS, max_frames=ACC_FRAMES)
    ref_acc = acc["PT_FP32"]
    rows = ["| Config | MOTA | \u0394MOTA vs PT FP32 | IDF1 | \u0394IDF1 | IDs | FP | FN |", "|---|---|---|---|---|---|---|---|"]
    rows += [f"| {k} | {m['MOTA']:.2f} | {m['MOTA'] - ref_acc['MOTA']:+.2f} | {m['IDF1']:.2f} | {m['IDF1'] - ref_acc['IDF1']:+.2f} | {m['IDs']} | {m['FP']} | {m['FN']} |" for k, m in acc.items()]
    display(Markdown("\n".join(rows)))
else:
    print("MOT17 not found - skipping MOTA / IDF1 accuracy evaluation (set MOT17_TRAIN to enable it)")
    for _k in ("PT_FP32", "OV_FP32", "OV_FP16"):
        acc[_k] = None

## Run FP32 and FP16 on a video
[back to top ⬆️](#Table-of-contents:)

The first `VIDEO_SECONDS` (20 s) of the free-license sample clip
(CC BY-SA 4.0, "Pro dog walker - Tokyo - 2024 Nov 1" by User:Nesnad, Wikimedia Commons). PyTorch FP32 tracks the same
frames first and every OpenVINO run is scored **against those reference tracks** (100 = identical). Set
`VIDEO_SECONDS = None` for the whole video.


In [ ]:
VIDEO_SECONDS = 20  # every video run in this notebook is clipped to the first 20 s; None = whole video
_cap = cv2.VideoCapture(str(U.VIDEO_1))
VIDEO_MAX_FRAMES = int(VIDEO_SECONDS * _cap.get(cv2.CAP_PROP_FPS)) if VIDEO_SECONDS else None
_cap.release()

# PyTorch FP32 reference tracks on the same frames
det_pt_run = U.torch_runner(det_pt, "cpu", torch.float32)
reid_pt_run = U.torch_runner(reid_pt, "cpu", torch.float32)
ART = Path("artifacts")  # tracking output videos land here (gitignored; upload as PR attachments)
(ART / "video").mkdir(parents=True, exist_ok=True)
pt_ref = U.track_video(det_pt_run, reid_pt_run, cfg, "PT_FP32", video=U.VIDEO_1, max_frames=VIDEO_MAX_FRAMES, write_mp4=False)
pt_ref_tracks = pt_ref["tracks"]
vid = {}


def video_run(tag: str, det_run, reid_run) -> None:
    r = U.track_video(det_run, reid_run, cfg, tag, video=U.VIDEO_1, out_dir=ART / "video", max_frames=VIDEO_MAX_FRAMES)
    r["stats"].update(U.video_agreement(pt_ref_tracks[: len(r["tracks"])], r["tracks"]))
    vid[tag] = r["stats"]
    s = r["stats"]
    print(f"{tag}: {s['frames']} frames, {s['unique_ids']} unique IDs | detector {s['detector_ms_per_frame']:.1f} + ReID {s['reid_ms_per_frame']:.1f} ms/frame | "
          f"{s['pipeline_fps']:.1f} FPS end to end | IDF1 vs PT FP32 {s['IDF1_vs_ref']:.1f}, MOTA vs PT FP32 {s['MOTA_vs_ref']:.1f}")


for p in ("FP32", "FP16"):
    video_run(f"OV_{p}", det_ov[p], reid_ov[p])
display(Image(U.video_contact_sheet(vid["OV_FP16"]["mp4"], OUT / "images/video_ov_fp16_sheet.jpg"), width=1000))

## Optimize models using NNCF Post-training Quantization API
[back to top ⬆️](#Table-of-contents:)

[NNCF](https://github.com/openvinotoolkit/nncf) provides a suite of advanced algorithms for neural network inference
optimization in OpenVINO with minimal accuracy drop. We use 8-bit quantization in post-training mode (without the
fine-tuning pipeline) for **both networks**. The tracker is not a network and is unchanged.

The optimization process contains the following steps:

1. Create calibration Datasets: with MOT17 available, **train-half** frames (disjoint from the val-half used for
   accuracy) for the detector and batches of 8 ground-truth person crops for the ReID; otherwise frames of the
   sample video and detector crops from them.
2. Run `nncf.quantize` on each **FP32 IR**.
3. Validate: ReID embedding fidelity (cosine vs FP32, rank-1 / mAP with MOT17), then MOTA / IDF1 of the whole
   tracker when MOT17 is available. If the accuracy drops by **more than 2 points**, run
   `nncf.quantize_with_accuracy_control` for the detector.
4. Serialize the OpenVINO IRs with `openvino.save_model`.

| Option | Detector | ReID | Why |
|---|---|---|---|
| `preset` | `MIXED` | `MIXED` | symmetric weights, asymmetric activations (SiLU / ReLU outputs are not zero-centred) |
| `ignored_scope` | box-decode tail (grid add, stride mul, `exp`, concat) | `heads.*`: GeM pooling (clamp → x³ → mean → x^⅓) + BN neck | the decode keeps box coordinates exact; quantizing x³ in GeM makes the embeddings drift (cosine vs FP32 0.71 instead of 0.95), and BoT-SORT gates on absolute similarity |
| `subset_size` | 300 frames | 300 batches | calibration samples |
| `fast_bias_correction` | `True` | `True` | full BiasCorrection caches all activations (~27 GB for the detector at 800×1440) |

Please select below whether you would like to run quantization to improve model inference speed.

In [ ]:
int8_det_path, int8_reid_path = ir("INT8", DET_NAME), ir("INT8", REID_NAME)
int8_det_ac_path = ir("INT8_accuracy_control", DET_NAME)

to_quantize = widgets.Checkbox(value=True, description="Quantization")
quantization_mode = widgets.Dropdown(
    options=[
        ("Default quantization (accuracy control only if ΔMOTA > 2)", "default"),
        ("Quantization with accuracy control (detector)", "accuracy_control"),
    ],
    value="default",
    description="Mode",
)

widgets.VBox([to_quantize, quantization_mode])

Let's load `skip magic` extension to skip quantization if `to_quantize` is not selected

In [ ]:
if not Path("skip_kernel_extension.py").exists():
    try:
        import requests

        r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/skip_kernel_extension.py", timeout=10)
        r.raise_for_status()
        open("skip_kernel_extension.py", "w").write(r.text)
    except Exception:
        open("skip_kernel_extension.py", "w").write(
            "def skip(line, cell=None):\n    if eval(line):\n        return\n    get_ipython().ex(cell)\n\n\n"
            "def load_ipython_extension(shell):\n    shell.register_magic_function(skip, 'line_cell')\n"
        )

%load_ext skip_kernel_extension

Prepare the calibration datasets. With MOT17 available: 300 evenly spaced MOT17 **train-half** frames (disjoint from
the val-half used for accuracy) for the detector, and 300 batches of 8 ground-truth person crops (pedestrians with
visibility ≥ 0.25) for the ReID. Without MOT17: frames of the sample video and detector crops from them. Both are
wrapped into `nncf.Dataset` objects.


In [ ]:
%%skip not $to_quantize.value

N_CALIB = 300
if MOT17_OK:
    train_half = U.train_half_frames()
    image_files = [train_half[i] for i in np.linspace(0, len(train_half) - 1, N_CALIB).round().astype(int)]

    def transform_fn(image_path: Path) -> np.ndarray:
        x, _ = U.preprocess(cv2.imread(str(image_path)), cfg)
        return x

    reid_calibration_items = list(U.reid_gt_crops(image_files, batch=cfg["reid"]["batch"], size_hw=tuple(cfg["reid"]["input_hw"])))
else:
    print("MOT17 not found - calibrating on sample-video frames and detector crops instead")
    _calib_frames = U.video_frames(U.VIDEO_1, N_CALIB)
    image_files = _calib_frames

    def transform_fn(image: np.ndarray) -> np.ndarray:
        x, _ = U.preprocess(image, cfg)
        return x

    reid_calibration_items = list(U.reid_det_crops(_calib_frames, det_pt_run, batch=cfg["reid"]["batch"],
                                                   size_hw=tuple(cfg["reid"]["input_hw"]), cfg=cfg))

det_fp32_model = core.read_model(ir("FP32", DET_NAME))
_ops = det_fp32_model.get_ordered_ops()
_t = max(i for i, o in enumerate(_ops) if o.get_type_name() == "Transpose")
det_ignored = [o.get_friendly_name() for o in _ops[_t:] if o.get_type_name() not in ("Constant", "Result")]
reid_ignored = [o.get_friendly_name() for o in core.read_model(ir("FP32", REID_NAME)).get_ordered_ops()
                if o.get_friendly_name().startswith("__module.heads.") and o.get_type_name() not in ("Constant", "Result")]
print(f"{len(image_files)} detector calibration frames, {len(reid_calibration_items)} ReID calibration batches | "
      f"ops kept in float: detector {len(det_ignored)} (box decode), ReID {len(reid_ignored)}")

The `nncf.quantize` function provides an interface for model quantization. It requires an instance of the OpenVINO
Model and a quantization dataset. Optional parameters configure the quantization process: number of samples, preset,
ignored scope, and so on.

For **accuracy control** on the detector, `validation_fn` runs the *whole* BoT-SORT pipeline (candidate detector →
NMS → INT8 ReID → tracker) and returns a metric: MOTA on a MOT17 val-half subset when the dataset is available, or
the video MOTA against the PyTorch FP32 reference tracks otherwise. NNCF compiles the candidate models on **CPU**
for this, so the subset is kept short.

>**Note**: Model post-training quantization is a time-consuming process. It takes a few minutes (ReID ~2 min,
>detector ~5 min); accuracy control takes longer.


In [ ]:
%%skip not $to_quantize.value

import nncf
import openvino as ov

MAX_DROP = 2.0  # accuracy points (absolute)


def save_int8(model: ov.Model, path: Path, info: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    model.set_rt_info(tracking_config, ["tracking_config"])
    ov.save_model(model, str(path))
    json.dump(cfg, open(path.parent / "botsort_config.json", "w"), indent=2)
    info["fake_quantize_nodes"] = sum(o.get_type_name() == "FakeQuantize" for o in model.get_ordered_ops())
    json.dump(info, open(path.with_suffix(".quantization_info.json"), "w"), indent=2)


def quantize_reid(out_path: Path) -> ov.Model:
    t = time.perf_counter()
    quantized_reid_model = nncf.quantize(
        core.read_model(ir("FP32", REID_NAME)),
        nncf.Dataset(reid_calibration_items),
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(reid_calibration_items),
        fast_bias_correction=True,
        ignored_scope=nncf.IgnoredScope(names=reid_ignored, validate=False),
    )
    print(f"Quantized ReID model will be saved to {out_path}")
    save_int8(quantized_reid_model, out_path, {"network": "reid", "preset": "mixed", "ignored_scope": reid_ignored, "time_s": time.perf_counter() - t})
    return quantized_reid_model


def video_validation_fn(compiled_model, validation_subset) -> float:
    out = compiled_model.output(0)
    run = lambda x: np.asarray(compiled_model(x)[out])  # noqa: E731
    r = U.track_video(run, reid_int8_run, cfg, "nncf_ac", video=U.VIDEO_1, max_frames=60, write_mp4=False)
    score = U.video_agreement(pt_ref_tracks[: len(r["tracks"])], r["tracks"])
    print(f"[NNCF] video MOTA vs PT FP32 = {score['MOTA_vs_ref']:.2f}", flush=True)
    return score["MOTA_vs_ref"]


def quantize_detector(use_accuracy_control: bool, out_path: Path, reid_run=None) -> ov.Model:
    ov_model = core.read_model(ir("FP32", DET_NAME))  # FP32 IR
    quantization_dataset = nncf.Dataset(image_files, transform_fn)
    calibration_dataset = quantization_dataset
    PRESET = nncf.QuantizationPreset.MIXED
    FAST_BIAS_CORRECTION = True
    IGNORED_SCOPE = nncf.IgnoredScope(names=det_ignored, validate=False)

    # nncf with accuracy control
    if use_accuracy_control:
        DROP_TYPE = nncf.DropType.ABSOLUTE
        CALIB_SIZE = len(image_files)
        if MOT17_OK:
            validation_items, nncf_validation_fn = U.accuracy_control_validation(reid_run, cfg, seqs=("MOT17-09-FRCNN",), frames=150)
            validation_dataset = nncf.Dataset(validation_items)  # one item = one MOT17 sequence
        else:
            validation_items = _calib_frames[:60]
            nncf_validation_fn = video_validation_fn
            validation_dataset = nncf.Dataset(validation_items)

    t = time.perf_counter()
    if not use_accuracy_control:
        quantized_det_model = nncf.quantize(
            ov_model,
            quantization_dataset,
            preset=PRESET,
            subset_size=len(image_files),
            fast_bias_correction=FAST_BIAS_CORRECTION,
            ignored_scope=IGNORED_SCOPE,
        )
    else:
        quantized_det_model = nncf.quantize_with_accuracy_control(
            ov_model,
            calibration_dataset=calibration_dataset,
            validation_dataset=validation_dataset,
            validation_fn=nncf_validation_fn,
            max_drop=MAX_DROP,
            drop_type=DROP_TYPE,
            preset=PRESET,
            subset_size=CALIB_SIZE,
            fast_bias_correction=FAST_BIAS_CORRECTION,
            ignored_scope=IGNORED_SCOPE,
        )
    print(f"Quantized detector model will be saved to {out_path}")
    save_int8(quantized_det_model, out_path, {"network": "detector", "preset": "mixed", "accuracy_control": use_accuracy_control,
                                              "ignored_scope": det_ignored, "time_s": time.perf_counter() - t})
    return quantized_det_model


if not int8_reid_path.exists():
    quantize_reid(int8_reid_path)
reid_int8_run = U.ov_runner(core, int8_reid_path, device.value)
use_accuracy_control = quantization_mode.value == "accuracy_control"
int8_det_path = int8_det_ac_path if use_accuracy_control else int8_det_path
if not int8_det_path.exists():
    quantize_detector(use_accuracy_control, int8_det_path, reid_int8_run)
for p in (int8_det_path, int8_reid_path):
    print(f"INT8 IR {p} ({p.with_suffix('.bin').stat().st_size / 1e6:.0f} MB) | FakeQuantize nodes: {sum(o.get_type_name() == 'FakeQuantize' for o in core.read_model(p).get_ordered_ops())}")

### Validate Quantized model inference
[back to top ⬆️](#Table-of-contents:)

First the **ReID embedding fidelity**: with MOT17, on ground-truth crops (query = crops at frame *f*, gallery = crops
at frame *f* + 15, rank-1 / mAP); without it, the embedding cosine against FP32 on detector crops. Then the detector
check and, when MOT17 is available, MOTA / IDF1 of the whole INT8 pipeline. If the accuracy drop vs OV FP32 is larger
than `MAX_DROP`, the next cell falls back to `nncf.quantize_with_accuracy_control` for the detector automatically.


In [ ]:
%%skip not $to_quantize.value

if MOT17_OK:
    fid = U.reid_fidelity({"OpenVINO FP32 (f32)": reid_ov["FP32"], "OpenVINO FP16": reid_ov["FP16"], "OpenVINO INT8": reid_int8_run},
                          pairs_per_seq=6 if SETTINGS["full_evaluation"] else 2)
    rows = ["| ReID precision | rank-1 (%) | mAP (%) | embedding cosine vs FP32 (mean / min) |", "|---|---|---|---|"]
    for k, v in fid.items():
        cos = "-" if v["cos_mean"] is None else f"{v['cos_mean']:.5f} / {v['cos_min']:.5f}"
        rows.append(f"| {k} | {v['rank1']:.2f} | {v['mAP']:.2f} | {cos} |")
    display(Markdown(f"{next(iter(fid.values()))['crops']} crops\n\n" + "\n".join(rows)))
else:
    _pairs = list(U.reid_det_crops(_calib_frames[:20], det_pt_run, batch=cfg["reid"]["batch"],
                                   size_hw=tuple(cfg["reid"]["input_hw"]), cfg=cfg))[:20]
    _ref = U.normalize_feats(np.concatenate([reid_ov["FP32"](b)[:len(b)] for b in _pairs]))
    rows = ["| ReID precision | embedding cosine vs FP32 (mean / min) |", "|---|---|"]
    for k, run in (("OpenVINO FP32 (f32)", reid_ov["FP32"]), ("OpenVINO FP16", reid_ov["FP16"]), ("OpenVINO INT8", reid_int8_run)):
        cos = np.concatenate([np.sum(a * b, 1) for a, b in zip(_ref, [U.normalize_feats(np.asarray(run(b))[:len(b)]) for b in _pairs])])
        rows.append(f"| {k} | {cos.mean():.5f} / {cos.min():.5f} |")
    display(Markdown(f"{2 * len(_pairs) * cfg['reid']['batch']} detector crops (no MOT17)\n\n" + "\n".join(rows)))

det_int8_run = U.ov_runner(core, int8_det_path, device.value)
y8, ms8 = U.timeit(det_int8_run, x0, n=10)
d8 = U.postprocess(y8, r0, cfg)
print(f"OV INT8 detector: {ms8:.1f} ms | cosine vs PT FP32: raw {U.cosine(y8, det_ref):.5f}, scores {U.score_cosine(y8, det_ref):.5f} | {len(d8)} detections (PT FP32 {len(d_ref)})")
cv2.imwrite(str(OUT / "images/ov_int8_detections.jpg"), U.draw_dets(img0, d8))
display(Image(str(OUT / "images/ov_int8_detections.jpg"), width=900))
int8_tag = "OV_INT8_AC" if int8_det_path == int8_det_ac_path else "OV_INT8"
if MOT17_OK:
    acc[int8_tag] = U.mot_eval(det_int8_run, reid_int8_run, cfg, int8_tag, seqs=ACC_SEQS, max_frames=ACC_FRAMES)
    drop = acc["OV_FP32"]["MOTA"] - acc[int8_tag]["MOTA"]
    print(f"{int8_tag}: MOTA {acc[int8_tag]['MOTA']:.2f} (drop vs OV FP32 {drop:+.2f}) | IDF1 {acc[int8_tag]['IDF1']:.2f} | IDs {acc[int8_tag]['IDs']}")
else:
    acc[int8_tag] = None
    drop = 0.0
    print("MOT17 not available - skipping MOTA / IDF1; the video run below scores INT8 against the PyTorch FP32 reference")

In [ ]:
%%skip not $to_quantize.value

if drop > MAX_DROP and int8_tag == "OV_INT8":
    print(f"accuracy drop {drop:.2f} > {MAX_DROP} -> nncf.quantize_with_accuracy_control (detector)")
    if not int8_det_ac_path.exists():
        quantize_detector(True, int8_det_ac_path, reid_int8_run)
    int8_det_path, int8_tag = int8_det_ac_path, "OV_INT8_AC"
    det_int8_run = U.ov_runner(core, int8_det_path, device.value)
    if MOT17_OK:
        acc[int8_tag] = U.mot_eval(det_int8_run, reid_int8_run, cfg, int8_tag, seqs=ACC_SEQS, max_frames=ACC_FRAMES)
        print(f"{int8_tag}: MOTA {acc[int8_tag]['MOTA']:.2f} (drop {acc['OV_FP32']['MOTA'] - acc[int8_tag]['MOTA']:+.2f})")
    if acc.get(int8_tag) is not None and acc["OV_FP32"]["MOTA"] - acc[int8_tag]["MOTA"] > MAX_DROP:
        raise RuntimeError(f"accuracy drop {acc['OV_FP32']['MOTA'] - acc[int8_tag]['MOTA']:.2f} still exceeds the {MAX_DROP} limit after accuracy control")
elif MOT17_OK or drop > MAX_DROP:
    print(f"accuracy drop {drop:+.2f} <= {MAX_DROP}: plain nncf.quantize() meets the criterion, accuracy control is not needed")

### Run INT8 on the video
[back to top ⬆️](#Table-of-contents:)

In [ ]:
%%skip not $to_quantize.value

video_run(int8_tag, det_int8_run, reid_int8_run)
display(Image(U.video_contact_sheet(vid[int8_tag]["mp4"], OUT / "images/video_ov_int8_sheet.jpg"), width=1000))

## Compare performance of the models with benchmark_app
[back to top ⬆️](#Table-of-contents:)

Finally, use the OpenVINO [Benchmark Tool](https://docs.openvino.ai/2026/learn-openvino/openvino-samples/benchmark-tool.html)
to measure the **average latency** and **throughput** of both networks at FP32 (`-infer_precision f32`), FP16 and
INT8, in two modes:
* `-hint latency -api sync`: 1 request;
* `-hint tput`: async, with the number of requests chosen by the plugin. 

`-ip f32` feeds f32 input like PyTorch. `-b` must equal the model batch (1 for the detector, 8 for the ReID); without
an explicit batch, `-hint tput` on GPU may enable automatic batching, which PyTorch does not mirror. For the ReID,
FPS = crops per second. Every run is preceded by a 10 s cool-down.

> **Note**: For more accurate performance, run `benchmark_app` in a terminal after closing other applications, e.g.
> `benchmark_app -m <model.xml> -d GPU -t 30 -b 1 -shape "[1,3,800,1440]" -hint tput -ip f32`.

In [ ]:
device

In [ ]:
BENCH_TIME = SETTINGS["bench_time_s"]
COOLDOWN_S = SETTINGS.get("cooldown_s", 10)
BENCH = OUT / "benchmark"
BENCH.mkdir(parents=True, exist_ok=True)
FP32_FLAGS = "-infer_precision f32" if device.value.startswith("GPU") else ""
# benchmark_app ships with the openvino pip package; use the one next to this kernel's python if it is not on PATH
benchmark_app = shutil.which("benchmark_app") or str(Path(sys.executable).parent / "benchmark_app")
NETS = {"detector": (DET_NAME, 1, "[" + ",".join(map(str, DET_SHAPE)) + "]"), "reid": (REID_NAME, cfg["reid"]["batch"], "[" + ",".join(map(str, REID_SHAPE)) + "]")}

In [ ]:
# Inference FP32 models (OpenVINO IR, executed in f32)
for net, (name, b, shape) in NETS.items():
    xml = ir("FP32", name)
    time.sleep(COOLDOWN_S)
    !$benchmark_app -m $xml -d $device.value -b $b -hint latency -api sync -ip f32 {FP32_FLAGS} -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_OV_FP32_sync.log 2>&1
    time.sleep(COOLDOWN_S)
    !$benchmark_app -m $xml -d $device.value -b $b -hint tput -ip f32 {FP32_FLAGS} -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_OV_FP32_tput.log 2>&1
    for mode in ("sync", "tput"):
        r = U.parse_benchmark_log(open(BENCH / f"{net}_OV_FP32_{mode}.log").read())
        print(f"{net:8s} OV FP32 {mode}: Average {r['avg_ms']:.2f} ms | Throughput {r['fps']:.1f} FPS | {r['api']} nireq={r['nireq']}")

In [ ]:
# Inference FP16 models (OpenVINO IR)
for net, (name, b, shape) in NETS.items():
    xml = ir("FP16", name)
    time.sleep(COOLDOWN_S)
    !$benchmark_app -m $xml -d $device.value -b $b -hint latency -api sync -ip f32  -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_OV_FP16_sync.log 2>&1
    time.sleep(COOLDOWN_S)
    !$benchmark_app -m $xml -d $device.value -b $b -hint tput -ip f32  -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_OV_FP16_tput.log 2>&1
    for mode in ("sync", "tput"):
        r = U.parse_benchmark_log(open(BENCH / f"{net}_OV_FP16_{mode}.log").read())
        print(f"{net:8s} OV FP16 {mode}: Average {r['avg_ms']:.2f} ms | Throughput {r['fps']:.1f} FPS | {r['api']} nireq={r['nireq']}")

In [ ]:
if int8_det_path.exists() and int8_reid_path.exists():
    # Inference INT8 models (OpenVINO IR)
    for net, xml in (("detector", int8_det_path), ("reid", int8_reid_path)):
        b, shape = NETS[net][1], NETS[net][2]
        time.sleep(COOLDOWN_S)
        !$benchmark_app -m $xml -d $device.value -b $b -hint latency -api sync -ip f32 -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_{int8_tag}_sync.log 2>&1
        time.sleep(COOLDOWN_S)
        !$benchmark_app -m $xml -d $device.value -b $b -hint tput -ip f32 -shape "$shape" -t $BENCH_TIME > {BENCH}/{net}_{int8_tag}_tput.log 2>&1
        for mode in ("sync", "tput"):
            r = U.parse_benchmark_log(open(BENCH / f"{net}_{int8_tag}_{mode}.log").read())
            print(f"{net:8s} {int8_tag} {mode}: Average {r['avg_ms']:.2f} ms | Throughput {r['fps']:.1f} FPS | {r['api']} nireq={r['nireq']}")

## Accuracy and performance: OpenVINO vs PyTorch
[back to top ⬆️](#Table-of-contents:)

All numbers come from the same machine with the same settings. The PyTorch FP32 baseline is measured in this notebook
with the same runners (CPU); accuracy is MOT17 val-half when available (identical host pipeline and tracker config);
performance is `benchmark_app` for OpenVINO and the timing-equivalent PyTorch measurement. The **network time per
frame** = detector latency + (ReID batches per frame, measured during the video run) × ReID batch latency.


In [ ]:
perf = {}
det_pt_ms, _ = U.timeit(det_pt_run, x0, n=10)
reid_pt_ms, _ = U.timeit(reid_pt_run, crops0, n=10)
perf[("detector", "PT_FP32", "sync")] = {"avg_ms": det_pt_ms, "fps": 1000 / det_pt_ms}
perf[("reid", "PT_FP32", "sync")] = {"avg_ms": reid_pt_ms, "fps": 1000 * cfg["reid"]["batch"] / reid_pt_ms}
ov_tags = ["OV_FP32", "OV_FP16"] + ([int8_tag] if "int8_tag" in globals() else [])
for tag in ov_tags:
    for net in NETS:
        f = BENCH / f"{net}_{tag}_sync.log"
        if f.exists():
            r = U.parse_benchmark_log(f.read_text())
            perf[(net, tag, "sync")] = {"avg_ms": r["avg_ms"], "fps": r["fps"], "nireq": r["nireq"]}
        f = BENCH / f"{net}_{tag}_tput.log"
        if f.exists():
            r = U.parse_benchmark_log(f.read_text())
            perf[(net, tag, "tput")] = {"avg_ms": r["avg_ms"], "fps": r["fps"], "nireq": r["nireq"]}

names = {"PT_FP32": "PyTorch FP32 (CPU)", "OV_FP32": "OpenVINO FP32 (f32)", "OV_FP16": "OpenVINO FP16",
         "OV_INT8": "OpenVINO INT8 (NNCF)", "OV_INT8_AC": "OpenVINO INT8 (NNCF, accuracy control)"}
table = []
for c in [k for k in names if k in acc or k == "PT_FP32"]:
    ds, rsync, dt = perf.get(("detector", c, "sync")), perf.get(("reid", c, "sync")), perf.get(("detector", c, "tput"))
    rs = (acc.get(c) or {}).get("reid_batches_per_frame") or vid.get(c, {}).get("reid_batches_per_frame", 0)
    frame_ms = ds["avg_ms"] + rs * rsync["avg_ms"] if ds and rsync else None
    a = acc.get(c)
    table.append({"config": names[c], "MOTA": a and round(a["MOTA"], 2), "IDF1": a and round(a["IDF1"], 2), "IDs": a and a["IDs"],
                  "detector_sync_avg_ms": ds and ds["avg_ms"], "detector_tput_fps": dt and dt["fps"],
                  "reid_sync_avg_ms_per_batch": perf.get(("reid", c, "sync"), {}).get("avg_ms"), "reid_tput_crops_per_s": perf.get(("reid", c, "tput"), {}).get("fps"),
                  "reid_batches_per_frame": round(rs, 2), "network_ms_per_frame": frame_ms and round(frame_ms, 1), "network_fps": frame_ms and round(1000 / frame_ms, 1),
                  "video_IDF1_vs_PT_FP32": vid.get(c, {}).get("IDF1_vs_ref")})
base = table[0]["network_ms_per_frame"]
fm = lambda v, p=1: "-" if v is None else (f"{v:.{p}f}" if isinstance(v, float) else str(v))  # noqa: E731
acc_cols = "MOTA | IDF1 | IDs | " if MOT17_OK else ""
rows = [f"| Config (detector + ReID) | {acc_cols}detector AVG ms | detector tput FPS | ReID AVG ms / batch | ReID tput crops/s | network ms / frame (FPS) | speed-up vs PT FP32 | video IDF1 vs PT FP32 |",
        "|" + "---|" * (10 if MOT17_OK else 7)]
for r in table:
    acc_part = f"{fm(r['MOTA'])} | {fm(r['IDF1'])} | {fm(r['IDs'], 0)} | " if MOT17_OK else ""
    rows.append(f"| {r['config']} | {acc_part}{fm(r['detector_sync_avg_ms'])} | {fm(r['detector_tput_fps'])} | "
                f"{fm(r['reid_sync_avg_ms_per_batch'], 2)} | {fm(r['reid_tput_crops_per_s'], 0)} | {fm(r['network_ms_per_frame'])} ({fm(r['network_fps'])}) | "
                f"{fm(base / r['network_ms_per_frame'] if r['network_ms_per_frame'] else None, 2)}x | {fm(r['video_IDF1_vs_PT_FP32'])} |")
display(Markdown("\n".join(rows)))
json.dump({"table": table, "accuracy": acc, "video": vid, "benchmark": {"|".join(k): v for k, v in perf.items()}}, open(OUT / "ov_results.json", "w"), indent=2, default=str)
import csv

with open(OUT / "ov_results.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(table[0].keys()))
    w.writeheader()
    w.writerows(table)
print("saved", OUT / "ov_results.json", "and .csv")

In [ ]:
import matplotlib.pyplot as plt

panels = [("network_ms_per_frame", "network time per frame (ms) ↓"), ("reid_sync_avg_ms_per_batch", "ReID AVERAGE latency per batch of 8 (ms) ↓")]
if MOT17_OK:
    panels.append(("MOTA", "MOTA on MOT17 val-half (%) ↑"))
fig, axs = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4))
lbl = [r["config"].replace("OpenVINO", "OV").replace("PyTorch", "PT").replace(" (NNCF, accuracy control)", " AC").replace(" (NNCF)", "").replace(" (CPU)", "") for r in table]
col = ["#ef6c00", "#00838f", "#4dd0e1", "#7b1fa2", "#ab47bc"][: len(table)]
for ax, (key, title) in zip(np.atleast_1d(axs), panels):
    vals = [r[key] or 0 for r in table]
    b = ax.bar(lbl, vals, color=col, edgecolor="k", lw=0.5)
    ax.bar_label(b, fmt="%.1f", fontsize=8)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    if key == "MOTA":
        ax.set_ylim(min(vals) - 1.5, max(vals) + 0.8)
plt.tight_layout()
plt.savefig(OUT / "images/ov_results.png", dpi=130)
plt.show()